# Modelo Multimodal: MTDE-Net (Baseline vs. Optimizado)

Este cuaderno implementa de manera secuencial el entrenamiento y comparación de **MTDE-Net**:
1. **Fase A (Baseline):** Entrenamiento con la configuración inicial por defecto (sin sintonizar).
2. **Fase B (Optimizado):** Entrenamiento con la configuración de hiperparámetros campeona encontrada mediante optimización bayesiana con Optuna y semilla fija.

Al final del cuaderno, se genera de manera automática una **tabla comparativa de métricas** lista para incluir en el reporte final de tu tesis.

In [1]:
import sys
import random
from pathlib import Path

# Añadir el directorio raíz al path de Python para permitir importaciones correctas
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from src.models.mtde_net import MTDE_Net
from src.loaders.mtde_net_loader import MultimodalThermalDataset
from src.utils import SqrtScaledMSELoss, eval_mtde_net_metrics, split_by_sequence

### 1. Semilla Global y Reproducibilidad

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### 2. Carga del Dataset y Partición Estricta (Compartida)

In [3]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
time_scale = 30.0

train_ds_full = MultimodalThermalDataset(
    metadata_csv="../processed_data/metadata_train.csv", 
    is_train=True, 
    min_time_s=0.0,
    time_scale=time_scale
)
val_ds_full = MultimodalThermalDataset(
    metadata_csv="../processed_data/metadata_train.csv", 
    is_train=False, 
    min_time_s=0.0,
    time_scale=time_scale
)

# Forzar el mapeo de ruta para entorno notebook
train_ds_full.root = Path("../processed_data")
val_ds_full.root = Path("../processed_data")

t_idx, v_idx = split_by_sequence(train_ds_full.df)

print(f"Dataset cargado exitosamente. Particiones: {len(t_idx)} train / {len(v_idx)} val")

Dataset cargado exitosamente. Particiones: 1111 train / 329 val


--- 
## FASE A: Entrenamiento del Modelo Baseline (Sin Tunear)

Esta sección entrena el modelo con los hiperparámetros iniciales por defecto.

In [4]:
# 1. Configuración de Hiperparámetros Baseline
BASELINE_CONFIG = {
    "epochs": 120,
    "patience": 15,
    "min_delta": 1.0,
    "batch_size": 16,
    "lr": 0.0005,
    "weight_decay": 0.0005,
    "dropout": 0.2
}

set_seed(42)

# 2. Dataloaders específicos para Baseline
train_loader_base = DataLoader(Subset(train_ds_full, t_idx), batch_size=BASELINE_CONFIG["batch_size"], shuffle=True)
val_loader_base = DataLoader(Subset(val_ds_full, v_idx), batch_size=BASELINE_CONFIG["batch_size"])
train_eval_loader_base = DataLoader(Subset(train_ds_full, t_idx), batch_size=BASELINE_CONFIG["batch_size"])

# 3. Inicializar Modelo Baseline
model_base = MTDE_Net(dropout=BASELINE_CONFIG["dropout"]).to(dev)
crit = SqrtScaledMSELoss(scale=None)
opt = torch.optim.AdamW(model_base.parameters(), lr=BASELINE_CONFIG["lr"], weight_decay=BASELINE_CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[30, 60, 80], gamma=0.8)

print(f"Iniciando entrenamiento Baseline... Parámetros: {sum(p.numel() for p in model_base.parameters()):,}")

# 4. Ciclo de Entrenamiento
best_mae_base = float("inf")
no_imp = 0

for ep in range(1, BASELINE_CONFIG["epochs"] + 1):
    model_base.train()
    train_loss, n = 0.0, 0
    for x_img, x_tab, y in train_loader_base:
        x_img, x_tab, y = x_img.to(dev), x_tab.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        loss = crit(model_base(x_img, x_tab), y)
        loss.backward()
        opt.step()
        train_loss += loss.item() * x_img.size(0)
        n += x_img.size(0)

    scheduler.step()
    
    # Evaluar métricas unificadas en segundos reales
    val_m = eval_mtde_net_metrics(model_base, val_loader_base, dev, scale=time_scale)
    train_m = eval_mtde_net_metrics(model_base, train_eval_loader_base, dev, scale=time_scale)
    
    v_mae = val_m["mae"]
    is_best = v_mae < best_mae_base - BASELINE_CONFIG["min_delta"]
    if is_best: 
        best_mae_base, no_imp = v_mae, 0
        torch.save(model_base.state_dict(), "../MTDE_Net_baseline.pt")
        baseline_metrics = val_m # Guardar para comparación final
    else: 
        no_imp += 1
        
    print(f"Ep {ep:03d} | Loss: {train_loss/n:.4f} | TrMAE: {train_m['mae']:5.2f}s | ValMAE: {v_mae:5.2f}s | "
          f"RMSE: {val_m['rmse']:5.2f}s | R2: {val_m['r2']:.4f} | MAPE: {val_m['mape']:5.2f}% | "
          f"Acc60: {val_m['acc60']:.2f}% | Acc120: {val_m['acc120']:.2f}% {'*' if is_best else ''}")
    
    if no_imp >= BASELINE_CONFIG["patience"]:
        print(f"Early stop alcanzado. Mejor Val MAE Baseline: {best_mae_base:.2f}s")
        break

Iniciando entrenamiento Baseline... Parámetros: 1,325,421
Ep 001 | Loss: 1.4660 | TrMAE: 107.69s | ValMAE: 127.23s | RMSE: 172.82s | R2: -0.0988 | MAPE: 70.00% | Acc60: 39.51% | Acc120: 60.49% *
Ep 002 | Loss: 0.6853 | TrMAE: 54.34s | ValMAE: 65.13s | RMSE: 97.86s | R2: 0.6477 | MAPE: 42.95% | Acc60: 66.87% | Acc120: 83.59% *
Ep 003 | Loss: 0.3945 | TrMAE: 88.48s | ValMAE: 93.43s | RMSE: 126.50s | R2: 0.4113 | MAPE: 58.18% | Acc60: 47.11% | Acc120: 73.25% 
Ep 004 | Loss: 0.3765 | TrMAE: 96.65s | ValMAE: 119.91s | RMSE: 168.18s | R2: -0.0405 | MAPE: 72.95% | Acc60: 40.73% | Acc120: 59.27% 
Ep 005 | Loss: 0.3260 | TrMAE: 41.88s | ValMAE: 50.30s | RMSE: 74.22s | R2: 0.7973 | MAPE: 34.93% | Acc60: 68.39% | Acc120: 90.88% *
Ep 006 | Loss: 0.3856 | TrMAE: 48.87s | ValMAE: 54.52s | RMSE: 76.28s | R2: 0.7859 | MAPE: 44.46% | Acc60: 66.87% | Acc120: 89.36% 
Ep 007 | Loss: 0.2468 | TrMAE: 43.30s | ValMAE: 52.36s | RMSE: 69.47s | R2: 0.8225 | MAPE: 40.37% | Acc60: 63.53% | Acc120: 90.58% 
Ep 008 

--- 
## FASE B: Entrenamiento del Modelo Optimizado (Optuna Champion)

Esta sección entrena el modelo con los hiperparámetros óptimos encontrados en tu estudio determinista con Optuna.

In [11]:
# 1. Configuración de Hiperparámetros Optimizados (Actualizados según tu Optuna con metadata_train)
OPTIMIZED_CONFIG = {
    "epochs": 120,
    "patience": 15,
    "min_delta": 1.0,
    'lr': 0.0008180975206277762, 
    'weight_decay': 0.00012881632695032252, 
    'batch_size': 32, 
    'dropout': 0.10206052682964781
}

set_seed(42)

# 2. Dataloaders específicos para Optimizado
train_loader_opt = DataLoader(Subset(train_ds_full, t_idx), batch_size=OPTIMIZED_CONFIG["batch_size"], shuffle=True)
val_loader_opt = DataLoader(Subset(val_ds_full, v_idx), batch_size=OPTIMIZED_CONFIG["batch_size"])
train_eval_loader_opt = DataLoader(Subset(train_ds_full, t_idx), batch_size=OPTIMIZED_CONFIG["batch_size"])

# 3. Inicializar Modelo Optimizado
model_opt = MTDE_Net(dropout=OPTIMIZED_CONFIG["dropout"]).to(dev)
crit = SqrtScaledMSELoss(scale=None)
opt = torch.optim.AdamW(model_opt.parameters(), lr=OPTIMIZED_CONFIG["lr"], weight_decay=OPTIMIZED_CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[30, 60, 80], gamma=0.8)

print(f"Iniciando entrenamiento Optimizado... Parámetros: {sum(p.numel() for p in model_opt.parameters()):,}")

# 4. Ciclo de Entrenamiento
best_mae_opt = float("inf")
no_imp = 0

for ep in range(1, OPTIMIZED_CONFIG["epochs"] + 1):
    model_opt.train()
    train_loss, n = 0.0, 0
    for x_img, x_tab, y in train_loader_opt:
        x_img, x_tab, y = x_img.to(dev), x_tab.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        loss = crit(model_opt(x_img, x_tab), y)
        loss.backward()
        opt.step()
        train_loss += loss.item() * x_img.size(0)
        n += x_img.size(0)

    scheduler.step()
    
    # Evaluar métricas unificadas en segundos reales
    val_m = eval_mtde_net_metrics(model_opt, val_loader_opt, dev, scale=time_scale)
    train_m = eval_mtde_net_metrics(model_opt, train_eval_loader_opt, dev, scale=time_scale)
    
    v_mae = val_m["mae"]
    is_best = v_mae < best_mae_opt - OPTIMIZED_CONFIG["min_delta"]
    if is_best: 
        best_mae_opt, no_imp = v_mae, 0
        torch.save(model_opt.state_dict(), "../MTDE_Net_best.pt")
        optimized_metrics = val_m # Guardar para comparación final
    else: 
        no_imp += 1
        
    print(f"Ep {ep:03d} | Loss: {train_loss/n:.4f} | TrMAE: {train_m['mae']:5.2f}s | ValMAE: {v_mae:5.2f}s | "
          f"RMSE: {val_m['rmse']:5.2f}s | R2: {val_m['r2']:.4f} | MAPE: {val_m['mape']:5.2f}% | "
          f"Acc60: {val_m['acc60']:.2f}% | Acc120: {val_m['acc120']:.2f}% {'*' if is_best else ''}")
    
    if no_imp >= OPTIMIZED_CONFIG["patience"]:
        print(f"Early stop alcanzado. Mejor Val MAE Optimizado: {best_mae_opt:.2f}s")
        break

Iniciando entrenamiento Optimizado... Parámetros: 1,325,421
Ep 001 | Loss: 1.5682 | TrMAE: 80.60s | ValMAE: 73.44s | RMSE: 110.80s | R2: 0.5484 | MAPE: 77.58% | Acc60: 62.31% | Acc120: 82.07% *
Ep 002 | Loss: 0.7118 | TrMAE: 86.64s | ValMAE: 98.66s | RMSE: 131.58s | R2: 0.3631 | MAPE: 62.25% | Acc60: 41.95% | Acc120: 75.08% 
Ep 003 | Loss: 0.4093 | TrMAE: 73.90s | ValMAE: 77.44s | RMSE: 122.17s | R2: 0.4509 | MAPE: 43.64% | Acc60: 65.65% | Acc120: 76.60% 
Ep 004 | Loss: 0.3215 | TrMAE: 87.71s | ValMAE: 96.26s | RMSE: 123.25s | R2: 0.4412 | MAPE: 60.07% | Acc60: 42.86% | Acc120: 70.82% 
Ep 005 | Loss: 0.2623 | TrMAE: 71.99s | ValMAE: 90.51s | RMSE: 121.57s | R2: 0.4563 | MAPE: 59.67% | Acc60: 44.38% | Acc120: 75.99% 
Ep 006 | Loss: 0.2298 | TrMAE: 52.68s | ValMAE: 68.76s | RMSE: 102.05s | R2: 0.6169 | MAPE: 43.14% | Acc60: 65.65% | Acc120: 83.28% *
Ep 007 | Loss: 0.1627 | TrMAE: 41.85s | ValMAE: 36.29s | RMSE: 49.13s | R2: 0.9112 | MAPE: 33.05% | Acc60: 79.33% | Acc120: 97.26% *
Ep 008 

--- 
## 3. Tabla Comparativa de Resultados Finales

Esta celda autogenera una tabla markdown limpia comparando ambos escenarios listos para exportar.

In [12]:
comparative_data = {
    "Métrica": ["MAE (Error Absoluto Medio)", "RMSE (Error Cuadrático Medio)", "R² (Coeficiente de Det.)", "MAPE (Error Porcentual)", "Acc@60s (Exactitud 1 min)", "Acc@120s (Exactitud 2 min)"],
    "Baseline (Default)": [
        f"{baseline_metrics['mae']:.2f} s",
        f"{baseline_metrics['rmse']:.2f} s",
        f"{baseline_metrics['r2']:.4f}",
        f"{baseline_metrics['mape']:.2f} %",
        f"{baseline_metrics['acc60']:.2f} %",
        f"{baseline_metrics['acc120']:.2f} %"
    ],
    "Optimizado (Optuna)": [
        f"{optimized_metrics['mae']:.2f} s",
        f"{optimized_metrics['rmse']:.2f} s",
        f"{optimized_metrics['r2']:.4f}",
        f"{optimized_metrics['mape']:.2f} %",
        f"{optimized_metrics['acc60']:.2f} %",
        f"{optimized_metrics['acc120']:.2f} %"
    ]
}

df_comparison = pd.DataFrame(comparative_data)
from IPython.display import display, Markdown
display(Markdown("### Tabla Comparativa de MTDE-Net para Tesis"))
display(df_comparison)

### Tabla Comparativa de MTDE-Net para Tesis

,Métrica,Baseline (Default),Optimizado (Optuna)
0,MAE (Error Absoluto Medio),24.95 s,24.65 s
1,RMSE (Error Cuadrático Medio),34.63 s,33.98 s
2,R² (Coeficiente de Det.),0.9559,0.9575
3,MAPE (Error Porcentual),26.68 %,22.77 %
4,Acc@60s (Exactitud 1 min),93.31 %,91.49 %
5,Acc@120s (Exactitud 2 min),98.78 %,99.09 %
